# Cityscapes Dataset Analysis Report

This notebook summarizes the dataset analysis for the Cityscapes semantic segmentation project. The analysis was refreshed with:

```bash
make eda
```

The generated source report is `../docs/dataset_analysis.md`, and the generated figures are stored in `../reports/figures/`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path('..').resolve()
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'cityscapes'
REPORT_PATH = PROJECT_ROOT / 'docs' / 'dataset_analysis.md'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

print(f'Dataset root: {DATA_ROOT}')
print(f'EDA report:    {REPORT_PATH}')
print(f'Figures dir:   {FIGURES_DIR}')

## 1. Dataset Structure

The project expects the official Cityscapes layout below. The dataset is not committed to git because of licensing and size, but the local structure is available for analysis.

In [ ]:
for split in ['train', 'val', 'test']:
    image_count = len(list((DATA_ROOT / 'leftImg8bit' / split).rglob('*_leftImg8bit.png')))
    mask_count = len(list((DATA_ROOT / 'gtFine' / split).rglob('*_gtFine_labelIds.png')))
    print(f'{split:>5}: {image_count:4d} images | {mask_count:4d} label-id masks')

### Key Count Summary

| Split | Images | Label masks | Role |
|---|---:|---:|---|
| train | 2,975 | 2,975 | training set |
| val | 500 | 500 | validation and model comparison |
| test | 1,525 | 1,525 | final inference-style split |
| **total** | **5,000** | **5,000** | full local Cityscapes folder |

The `gtFine` tree also contains additional annotation files beyond `labelIds`, which is why the raw file count is larger than the number of masks used for training and evaluation.

## 2. Image Size Distribution

The sampled images are consistently 2048 x 1024 pixels. Training configs resize/crop them to fit GPU memory, usually with resize dimensions 512 x 1024 and crop size 512 x 512.

![Image size distribution](../reports/figures/image_size_distribution.png)

## 3. Class Distribution

Cityscapes segmentation is highly imbalanced. Large layout classes dominate the pixel count, while traffic participants and thin structures are rare.

In [ ]:
class_distribution = pd.DataFrame(
    [
        ('road', 70268061, 38.0954),
        ('sidewalk', 11384115, 6.1718),
        ('building', 43798798, 23.7452),
        ('wall', 602854, 0.3268),
        ('fence', 1259517, 0.6828),
        ('pole', 3194979, 1.7321),
        ('traffic light', 400655, 0.2172),
        ('traffic sign', 1106086, 0.5997),
        ('vegetation', 28825446, 15.6275),
        ('terrain', 1504880, 0.8159),
        ('sky', 5431163, 2.9445),
        ('person', 2327441, 1.2618),
        ('rider', 262899, 0.1425),
        ('car', 11670936, 6.3273),
        ('truck', 708579, 0.3842),
        ('bus', 580826, 0.3149),
        ('train', 101684, 0.0551),
        ('motorcycle', 371341, 0.2013),
        ('bicycle', 652639, 0.3538),
    ],
    columns=['class', 'pixels', 'percentage'],
)

class_distribution.sort_values('percentage', ascending=False).style.format(
    {'pixels': '{:,}', 'percentage': '{:.4f}%'}
)

![Class distribution](../reports/figures/class_distribution.png)

### Findings

- `road` alone accounts for **38.10%** of sampled pixels.
- `building` accounts for **23.75%**, and `vegetation` for **15.63%**.
- The rarest classes are `train`, `rider`, `motorcycle`, `traffic light`, and `bus`.
- This imbalance makes plain cross-entropy vulnerable to over-optimizing frequent classes.
- Dice loss and focal loss are useful comparisons because they reduce dependence on raw pixel frequency.

## 4. Category-Level Distribution

Grouping labels by scene category makes the imbalance easier to interpret. Flat surfaces and construction/nature classes dominate the visual field.

In [ ]:
category_distribution = pd.DataFrame(
    [
        ('flat', 81652176, 44.2672),
        ('construction', 45661169, 24.7549),
        ('nature', 30330326, 16.4434),
        ('vehicle', 14086005, 7.6366),
        ('sky', 5431163, 2.9445),
        ('object', 4701720, 2.5490),
        ('human', 2590340, 1.4043),
    ],
    columns=['category', 'pixels', 'percentage'],
)

category_distribution.style.format({'pixels': '{:,}', 'percentage': '{:.4f}%'})

![Category proportions](../reports/figures/category_pixel_proportions.png)

![Class pixels by category](../reports/figures/paper_class_pixels_by_category.png)

## 5. Qualitative Examples

The overlay figure checks that images and masks align spatially and that the label mapping is visually plausible. Rare-class examples are useful for manual inspection because rare classes drive many segmentation errors.

![Sample overlays](../reports/figures/sample_overlays.png)

![Rare class examples](../reports/figures/rare_classes_examples.png)

## 6. Additional Dataset Statistics

The project also generates paper-style summaries that help connect Cityscapes to common semantic-segmentation dataset diagnostics: annotation density, traffic participant complexity, and vehicle component area distributions.

![Annotation summary](../reports/figures/annotation_summary_table.png)

![Traffic participant complexity](../reports/figures/traffic_participant_complexity.png)

![Vehicle component area histogram](../reports/figures/vehicle_component_area_histogram.png)

## 7. Anomaly Checks

The refreshed EDA run reported:

| Check | Result |
|---|---:|
| missing images | 0 |
| missing masks | 0 |
| unreadable files | 0 |
| invalid mask values | 0 |
| mismatched image/mask dimensions | 0 |

No structural anomalies were found in the scanned files.

## 8. Modeling Implications

The data analysis directly informs the experiment design:

- **Class imbalance:** motivates Dice, focal, CE+Dice, and focal+Dice experiments.
- **Small objects:** traffic signs, traffic lights, poles, riders, and motorcycles require high-quality boundaries and enough augmentation.
- **Large image size:** resizing and cropping are necessary for practical GPU training.
- **Consistent dimensions:** simple resize/crop pipelines are safe because image/mask dimensions match.
- **Validation split:** the 500 validation images are the main comparison split for model selection and reporting.

Overall, the dataset is clean and well structured, but heavily imbalanced. The final training plan should judge models by mean IoU and mean Dice, not pixel accuracy alone, because pixel accuracy can look strong even when rare classes are weak.